# Stock Price Prediction using LSTM and Technical IndicatorsThis notebook builds an LSTM (Long Short-Term Memory) neural network to predict next-day stock closing prices, using a 60-day lookback window of price and technical indicator data.**Pipeline:**1. Fetch historical stock data via `yfinance`2. Engineer features: 50-day & 200-day moving averages, RSI (Relative Strength Index)3. Scale and reshape data into sequences for LSTM input4. Train an LSTM model with dropout regularisation5. Evaluate on a held-out test set and visualise predictions vs. actual prices**Disclaimer:** This is an educational project exploring time-series deep learning. It is *not* a trading system and should not be used to make real investment decisions — stock prices are influenced by far more than historical price patterns, and this model has no risk management, transaction cost modelling, or out-of-sample validation beyond a simple train/test split.

## 1. Setup & Imports

In [ ]:
!pip install yfinance -q

In [ ]:
import numpy as npimport pandas as pdimport yfinance as yfimport matplotlib.pyplot as pltfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.metrics import mean_squared_error, mean_absolute_errorfrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import LSTM, Dense, Dropoutimport warningswarnings.filterwarnings('ignore')np.random.seed(42)

## 2. ConfigurationAll key parameters are defined here so the notebook can be easily re-run for a different ticker, date range, or lookback window.

In [ ]:
TICKER = 'AAPL'START_DATE = '2015-01-01'END_DATE = '2023-01-01'LOOKBACK_WINDOW = 60      # days of history used to predict the next dayTRAIN_TEST_SPLIT = 0.8    # fraction of data used for trainingMA_SHORT_WINDOW = 20      # short moving average window (days)MA_LONG_WINDOW = 50       # long moving average window (days)RSI_PERIOD = 14           # RSI lookback period (days)EPOCHS = 100BATCH_SIZE = 32

## 3. Data Fetching

In [ ]:
def fetch_stock_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:    """Download historical OHLCV data for a given ticker from Yahoo Finance."""    stock_data = yf.download(ticker, start=start_date, end=end_date)    print(f"Fetched {len(stock_data)} rows of data for {ticker} ({start_date} to {end_date})")    return stock_datastock_data = fetch_stock_data(TICKER, START_DATE, END_DATE)stock_data.head()

## 4. Feature EngineeringWe compute two moving averages and the RSI momentum indicator, then drop any rows with `NaN` values introduced by the rolling-window calculations.

In [ ]:
def compute_rsi(series: pd.Series, period: int) -> pd.Series:    """Compute the Relative Strength Index using Wilder's exponential smoothing."""    delta = series.diff()    gain = delta.where(delta > 0, 0)    loss = -delta.where(delta < 0, 0)    avg_gain = gain.ewm(alpha=1 / period, min_periods=period).mean()    avg_loss = loss.ewm(alpha=1 / period, min_periods=period).mean()    rs = avg_gain / avg_loss    rsi = 100 - (100 / (1 + rs))    # Guard against division-by-zero when avg_loss is 0 (price only went up)    rsi = rsi.replace([np.inf, -np.inf], np.nan).fillna(50)    return rsidef engineer_features(stock_data: pd.DataFrame) -> pd.DataFrame:    """Add moving average and RSI features, then drop rows with NaNs from the rolling windows."""    df = stock_data.copy()    df['MA_short'] = df['Adj Close'].rolling(window=MA_SHORT_WINDOW).mean()    df['MA_long'] = df['Adj Close'].rolling(window=MA_LONG_WINDOW).mean()    df['RSI'] = compute_rsi(df['Adj Close'], RSI_PERIOD)    rows_before = len(df)    df = df.dropna()    print(f"Dropped {rows_before - len(df)} rows with NaN values from rolling-window warm-up")    print(f"{len(df)} rows remaining after feature engineering")    if df.empty:        raise ValueError("No valid rows remain after feature engineering — check the input data.")    return dfstock_data = engineer_features(stock_data)stock_data[['Adj Close', 'MA_short', 'MA_long', 'RSI']].tail()

### Visualise price and indicators

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)axes[0].plot(stock_data.index, stock_data['Adj Close'], label='Adj Close', color='black', linewidth=1)axes[0].plot(stock_data.index, stock_data['MA_short'], label=f'{MA_SHORT_WINDOW}-day MA', color='orange')axes[0].plot(stock_data.index, stock_data['MA_long'], label=f'{MA_LONG_WINDOW}-day MA', color='blue')axes[0].set_title(f'{TICKER} Price & Moving Averages')axes[0].set_ylabel('Price (USD)')axes[0].legend()axes[1].plot(stock_data.index, stock_data['RSI'], color='purple')axes[1].axhline(70, color='red', linestyle='--', linewidth=0.8, label='Overbought (70)')axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')axes[1].set_title('RSI')axes[1].set_ylabel('RSI')axes[1].legend()plt.tight_layout()plt.savefig('price_and_indicators.png', dpi=120)plt.show()

## 5. Prepare Sequences for LSTMThe LSTM needs 3D input of shape `(samples, timesteps, features)`. We scale all features to [0, 1] and build overlapping windows of `LOOKBACK_WINDOW` days, with the next day's `Adj Close` as the target.

In [ ]:
def prepare_sequences(stock_data: pd.DataFrame, feature_cols: list, target_col_idx: int, lookback: int):    """Scale features and build (X, y) sequences for LSTM training."""    scaler = MinMaxScaler(feature_range=(0, 1))    if stock_data[feature_cols].isnull().values.any():        raise ValueError("NaN values detected in feature columns prior to scaling.")    scaled_data = scaler.fit_transform(stock_data[feature_cols].values)    X, y = [], []    for i in range(lookback, len(scaled_data)):        X.append(scaled_data[i - lookback:i, :])        y.append(scaled_data[i, target_col_idx])    X, y = np.array(X), np.array(y)    if X.shape[0] == 0:        raise ValueError("Not enough rows to build a single sequence — reduce LOOKBACK_WINDOW or fetch more data.")    return X, y, scalerFEATURE_COLS = ['Adj Close', 'MA_short', 'MA_long', 'RSI']TARGET_COL_IDX = 0  # index of 'Adj Close' within FEATURE_COLSX, y, scaler = prepare_sequences(stock_data, FEATURE_COLS, TARGET_COL_IDX, LOOKBACK_WINDOW)print(f"X shape: {X.shape}  (samples, timesteps, features)")print(f"y shape: {y.shape}")

In [ ]:
split_idx = int(TRAIN_TEST_SPLIT * len(X))X_train, X_test = X[:split_idx], X[split_idx:]y_train, y_test = y[:split_idx], y[split_idx:]print(f"Train samples: {len(X_train)}  |  Test samples: {len(X_test)}")

## 6. Build the LSTM ModelA two-layer LSTM with dropout regularisation between layers to reduce overfitting on the relatively small dataset.

In [ ]:
def build_lstm_model(input_shape: tuple) -> Sequential:    model = Sequential([        LSTM(units=50, return_sequences=True, input_shape=input_shape),        Dropout(0.2),        LSTM(units=50),        Dropout(0.2),        Dense(units=1),    ])    model.compile(optimizer='adam', loss='mean_squared_error')    return modelmodel = build_lstm_model((X_train.shape[1], X_train.shape[2]))model.summary()

## 7. Train the Model

In [ ]:
history = model.fit(    X_train, y_train,    epochs=EPOCHS,    batch_size=BATCH_SIZE,    validation_split=0.1,    verbose=1,)

In [ ]:
plt.figure(figsize=(10, 4))plt.plot(history.history['loss'], label='Train Loss')plt.plot(history.history['val_loss'], label='Validation Loss')plt.title('Training Loss (MSE)')plt.xlabel('Epoch')plt.ylabel('Loss')plt.legend()plt.savefig('training_loss.png', dpi=120)plt.show()

## 8. Evaluate on Test SetWe inverse-transform predictions back to actual price scale and report standard regression error metrics alongside a visual comparison.

In [ ]:
def inverse_transform_target(scaled_values: np.ndarray, scaler: MinMaxScaler, target_col_idx: int, n_features: int) -> np.ndarray:    """Inverse-transform a single scaled column back to its original scale."""    dummy = np.zeros((len(scaled_values), n_features))    dummy[:, target_col_idx] = scaled_values    return scaler.inverse_transform(dummy)[:, target_col_idx]predicted_scaled = model.predict(X_test).flatten()predicted_price = inverse_transform_target(predicted_scaled, scaler, TARGET_COL_IDX, len(FEATURE_COLS))actual_price = inverse_transform_target(y_test, scaler, TARGET_COL_IDX, len(FEATURE_COLS))rmse = np.sqrt(mean_squared_error(actual_price, predicted_price))mae = mean_absolute_error(actual_price, predicted_price)mape = np.mean(np.abs((actual_price - predicted_price) / actual_price)) * 100print(f"RMSE: ${rmse:.2f}")print(f"MAE:  ${mae:.2f}")print(f"MAPE: {mape:.2f}%")

In [ ]:
plt.figure(figsize=(14, 5))plt.plot(actual_price, color='red', label='Actual Price')plt.plot(predicted_price, color='blue', label='Predicted Price')plt.title(f'{TICKER} — Predicted vs. Actual Closing Price (Test Set)')plt.xlabel('Time (test set index)')plt.ylabel('Price (USD)')plt.legend()plt.savefig('predictions_vs_actual.png', dpi=120)plt.show()

## 9. Save the ModelThe trained model is saved in Keras's native `.keras` format for later reuse (e.g. by `app.py` for serving predictions via an API).

In [ ]:
model.save('lstm_stock_model.keras')print("Model saved to lstm_stock_model.keras")

## Limitations & Next Steps- **No walk-forward validation**: a single train/test split likely overstates real-world performance compared to rolling-origin backtesting.- **No transaction costs or slippage modelling**: predicted price accuracy does not translate directly into trading profitability.- **Univariate target, despite multivariate input**: the model only predicts `Adj Close`; it doesn't model volatility or direction confidence.- **Possible improvements**: add walk-forward cross-validation, predict returns instead of raw price, incorporate volume-based features, compare against a naive baseline (e.g. last-value or ARIMA) to check the LSTM is adding genuine value.